# Week 4: Machine Learning Model Development and Evaluation
**Task:** Build and evaluate a classification model with cross-validation and hyperparameter tuning.
**Dataset:** Breast Cancer Wisconsin (Diagnostic) – UCI Machine Learning Repository (also bundled with scikit-learn).
**Goal:** Predict whether a tumour is *malignant* or *benign* from 30 numeric features computed from digitised images of fine-needle-aspirate (FNA) samples.

**Notebook flow:** 1) Load data → 2) EDA → 3) Preprocess & split → 4) Baseline model comparison (CV) → 5) Hyperparameter tuning (GridSearchCV) → 6) Final test evaluation → 7) Interpretation → 8) Conclusions.

In [1]:
import json, warnings, numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, RocCurveDisplay, ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
RS = 42
results = {}

## 1. Load the dataset

In [2]:
raw = load_breast_cancer(as_frame=True)
df = raw.frame.copy()
# In scikit-learn, 0 = malignant and 1 = benign. We flip it so that 1 = malignant (the class we most want to detect).
df['target'] = 1 - df['target']
X = df.drop(columns='target'); y = df['target']
print('Shape:', df.shape)
print('Missing values:', int(df.isna().sum().sum()))
print('Duplicate rows:', int(df.duplicated().sum()))
df.head()

Shape: (569, 31)
Missing values: 0
Duplicate rows: 0


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,1
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,1
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,1
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,1
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,1


## 2. Exploratory Data Analysis

In [3]:
counts = y.value_counts().sort_index()
print(counts.rename({0:'Benign',1:'Malignant'}))
print((counts/len(y)*100).round(1).rename({0:'Benign %',1:'Malignant %'}))
results['n_rows'], results['n_features'] = int(X.shape[0]), int(X.shape[1])
results['n_benign'], results['n_malignant'] = int(counts[0]), int(counts[1])
df.describe().T.head(10)

target
Benign       357
Malignant    212
Name: count, dtype: int64
target
Benign %       62.7
Malignant %    37.3
Name: count, dtype: float64


,count,mean,std,min,25%,50%,75%,max
mean radius,569.0,14.127292,3.524049,6.98100,11.70000,13.37000,15.78000,28.11000
mean texture,569.0,19.289649,4.301036,9.71000,16.17000,18.84000,21.80000,39.28000
mean perimeter,569.0,91.969033,24.298981,43.79000,75.17000,86.24000,104.10000,188.50000
mean area,569.0,654.889104,351.914129,143.50000,420.30000,551.10000,782.70000,2501.00000
mean smoothness,569.0,0.096360,0.014064,0.05263,0.08637,0.09587,0.10530,0.16340
mean compactness,569.0,0.104341,0.052813,0.01938,0.06492,0.09263,0.13040,0.34540
mean concavity,569.0,0.088799,0.079720,0.00000,0.02956,0.06154,0.13070,0.42680
mean concave points,569.0,0.048919,0.038803,0.00000,0.02031,0.03350,0.07400,0.20120
mean symmetry,569.0,0.181162,0.027414,0.10600,0.16190,0.17920,0.19570,0.30400
mean fractal dimension,569.0,0.062798,0.007060,0.04996,0.05770,0.06154,0.06612,0.09744


In [4]:
fig, ax = plt.subplots(1,2, figsize=(11,4))
sns.countplot(x=y.map({0:'Benign',1:'Malignant'}), ax=ax[0], palette=['#4C78A8','#E45756'])
ax[0].set_title('Class distribution'); ax[0].set_xlabel('')
top = X.corrwith(y).abs().sort_values(ascending=False).head(10)
sns.barplot(x=top.values, y=top.index, ax=ax[1], color='#4C78A8')
ax[1].set_title('Top 10 features by |correlation| with target'); ax[1].set_xlabel('|Pearson r|')
plt.tight_layout(); plt.savefig('fig_eda.png', dpi=150); plt.close()
results['top_corr'] = {k: round(float(v),3) for k,v in top.head(5).items()}
top

worst concave points    0.793566
worst perimeter         0.782914
mean concave points     0.776614
worst radius            0.776454
mean perimeter          0.742636
worst area              0.733825
mean radius             0.730029
mean area               0.708984
mean concavity          0.696360
worst concavity         0.659610
dtype: float64

In [5]:
corr = X.corr()
plt.figure(figsize=(9,7))
sns.heatmap(corr, cmap='coolwarm', center=0, xticklabels=False, yticklabels=False)
plt.title('Feature correlation heatmap (30 features)')
plt.tight_layout(); plt.savefig('fig_corr.png', dpi=150); plt.close()
hi = (corr.abs().where(np.triu(np.ones(corr.shape),1).astype(bool)) > 0.9).sum().sum()
print('Feature pairs with |r| > 0.9:', int(hi))
results['high_corr_pairs'] = int(hi)

Feature pairs with |r| > 0.9: 21


## 3. Preprocessing and train/test split
* No missing values or duplicates, so no imputation is needed.
* We use an **80/20 stratified split** so both sets keep the same benign/malignant ratio.
* Scaling (StandardScaler) is placed **inside a Pipeline** so it is fitted only on the training folds. This prevents data leakage during cross-validation.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RS)
print('Train:', X_train.shape, ' Test:', X_test.shape)
print('Train malignant %:', round(y_train.mean()*100,1), ' Test malignant %:', round(y_test.mean()*100,1))
results['n_train'], results['n_test'] = len(X_train), len(X_test)
results['train_mal_pct'], results['test_mal_pct'] = round(y_train.mean()*100,1), round(y_test.mean()*100,1)

Train: (455, 30)  Test: (114, 30)
Train malignant %: 37.4  Test malignant %: 36.8


## 4. Baseline model comparison with 5-fold stratified cross-validation
Four candidate algorithms with default-like settings are compared on the **training set only**.

In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RS)
models = {
 'Logistic Regression': Pipeline([('sc',StandardScaler()),('m',LogisticRegression(max_iter=5000, random_state=RS))]),
 'SVM (RBF)':           Pipeline([('sc',StandardScaler()),('m',SVC(probability=True, random_state=RS))]),
 'Decision Tree':       Pipeline([('sc',StandardScaler()),('m',DecisionTreeClassifier(random_state=RS))]),
 'Random Forest':       Pipeline([('sc',StandardScaler()),('m',RandomForestClassifier(random_state=RS))]),
}
scoring = ['accuracy','precision','recall','f1','roc_auc']
rows=[]
for n,m in models.items():
    s = cross_validate(m, X_train, y_train, cv=cv, scoring=scoring)
    rows.append({'Model':n, **{k: s[f'test_{k}'].mean() for k in scoring},
                 'F1 std': s['test_f1'].std()})
base = pd.DataFrame(rows).set_index('Model').round(4)
results['baseline'] = base.to_dict('index')
base

,accuracy,precision,recall,f1,roc_auc,F1 std
Model,,,,,,
Logistic Regression,0.9736,0.9771,0.9529,0.9640,0.9958,0.0207
SVM (RBF),0.9714,0.9769,0.9471,0.9610,0.9949,0.0082
Decision Tree,0.9231,0.8996,0.8941,0.8962,0.9172,0.0447
Random Forest,0.9648,0.9641,0.9412,0.9519,0.9889,0.0207


In [8]:
ax = base[['accuracy','precision','recall','f1','roc_auc']].plot(kind='bar', figsize=(10,4.5), ylim=(0.88,1.0), rot=15, colormap='tab10')
ax.set_title('Baseline models: mean 5-fold CV scores (training data)'); ax.set_ylabel('Score')
plt.legend(loc='lower right', ncol=5, fontsize=8); plt.tight_layout(); plt.savefig('fig_baseline.png', dpi=150); plt.close()

## 5. Hyperparameter tuning with GridSearchCV
We tune the two strongest families (Random Forest and SVM) using 5-fold stratified CV. The refit metric is **F1**, because it balances precision and recall. In a cancer-screening setting a missed malignant case (false negative) is the most costly error, so recall is also tracked closely.

In [9]:
rf_pipe = Pipeline([('sc',StandardScaler()),('m',RandomForestClassifier(random_state=RS))])
rf_grid = {'m__n_estimators':[100,300], 'm__max_depth':[None,5,10],
           'm__min_samples_split':[2,5], 'm__min_samples_leaf':[1,2,4], 'm__max_features':['sqrt','log2']}
gs_rf = GridSearchCV(rf_pipe, rf_grid, cv=cv, scoring=scoring, refit='f1', n_jobs=-1)
gs_rf.fit(X_train, y_train)
print('RF best params:', gs_rf.best_params_)
print('RF best CV F1 :', round(gs_rf.best_score_,4))
print('Combinations tried:', len(gs_rf.cv_results_['params']))

RF best params: {'m__max_depth': 5, 'm__max_features': 'sqrt', 'm__min_samples_leaf': 1, 'm__min_samples_split': 2, 'm__n_estimators': 100}
RF best CV F1 : 0.9553
Combinations tried: 72


In [10]:
svm_pipe = Pipeline([('sc',StandardScaler()),('m',SVC(probability=True, random_state=RS))])
svm_grid = {'m__C':[0.1,1,10,100], 'm__gamma':['scale',0.001,0.01,0.1], 'm__kernel':['rbf','linear']}
gs_svm = GridSearchCV(svm_pipe, svm_grid, cv=cv, scoring=scoring, refit='f1', n_jobs=-1)
gs_svm.fit(X_train, y_train)
print('SVM best params:', gs_svm.best_params_)
print('SVM best CV F1 :', round(gs_svm.best_score_,4))
print('Combinations tried:', len(gs_svm.cv_results_['params']))

SVM best params: {'m__C': 10, 'm__gamma': 'scale', 'm__kernel': 'rbf'}
SVM best CV F1 : 0.9674
Combinations tried: 32


In [11]:
def cvrow(gs):
    i = gs.best_index_
    return {k: round(float(gs.cv_results_[f'mean_test_{k}'][i]),4) for k in scoring} | {'F1 std': round(float(gs.cv_results_['std_test_f1'][i]),4)}
tuned = pd.DataFrame({'Random Forest (tuned)':cvrow(gs_rf),'SVM (tuned)':cvrow(gs_svm)}).T
results['tuned'] = tuned.to_dict('index')
results['rf_best_params'] = {k.replace('m__',''):v for k,v in gs_rf.best_params_.items()}
results['svm_best_params'] = {k.replace('m__',''):v for k,v in gs_svm.best_params_.items()}
results['rf_n_combos'] = len(gs_rf.cv_results_['params']); results['svm_n_combos'] = len(gs_svm.cv_results_['params'])
tuned

,accuracy,precision,recall,f1,roc_auc,F1 std
Random Forest (tuned),0.9670,0.9645,0.9471,0.9553,0.9882,0.0284
SVM (tuned),0.9758,0.9711,0.9647,0.9674,0.9935,0.0114


In [12]:
# Sensitivity of RF F1 to the main hyperparameters
res = pd.DataFrame(gs_rf.cv_results_)
fig, ax = plt.subplots(1,3, figsize=(13,3.8))
for a,p in zip(ax, ['param_m__n_estimators','param_m__max_depth','param_m__min_samples_leaf']):
    g = res.groupby(p)['mean_test_f1'].agg(['mean','max']).reset_index()
    g[p] = g[p].astype(str)
    a.plot(g[p], g['mean'], 'o-', label='mean over grid'); a.plot(g[p], g['max'], 's--', label='best')
    a.set_title(p.replace('param_m__','')); a.set_ylabel('CV F1')
ax[0].legend(); plt.tight_layout(); plt.savefig('fig_tuning.png', dpi=150); plt.close()

## 6. Final model and evaluation on the untouched test set
Model selection is based on **cross-validation only**. Tuned SVM has the highest CV F1 (0.967 vs 0.955), but the gap is within one standard deviation of the fold scores. We therefore evaluate both tuned models on the test set (used once) and present the **Random Forest** as the primary model for detailed analysis because it gives directly interpretable feature importances.

In [13]:
cands = {'Random Forest':gs_rf, 'SVM (RBF/linear)':gs_svm}
best_name = max(cands, key=lambda k: cands[k].best_score_)
print('Selected on CV F1:', best_name)
# Random Forest is kept as the primary model for its interpretability (feature importance); compare both on test
def evaluate(model, name):
    p = model.predict(X_test); pr = model.predict_proba(X_test)[:,1]
    return {'Model':name,'accuracy':accuracy_score(y_test,p),'precision':precision_score(y_test,p),
            'recall':recall_score(y_test,p),'f1':f1_score(y_test,p),'roc_auc':roc_auc_score(y_test,pr)}
untuned_rf = models['Random Forest'].fit(X_train,y_train)
test_tbl = pd.DataFrame([evaluate(untuned_rf,'Random Forest (default)'),
                         evaluate(gs_rf.best_estimator_,'Random Forest (tuned)'),
                         evaluate(gs_svm.best_estimator_,'SVM (tuned)')]).set_index('Model').round(4)
results['test'] = test_tbl.to_dict('index')
test_tbl

Selected on CV F1: SVM (RBF/linear)


,accuracy,precision,recall,f1,roc_auc
Model,,,,,
Random Forest (default),0.9737,1.0,0.9286,0.963,0.9929
Random Forest (tuned),0.9737,1.0,0.9286,0.963,0.9950
SVM (tuned),0.9737,1.0,0.9286,0.963,0.9927


In [14]:
final = gs_rf.best_estimator_
pred = final.predict(X_test); proba = final.predict_proba(X_test)[:,1]
cm = confusion_matrix(y_test, pred); tn, fp, fn, tp = cm.ravel()
results['cm'] = dict(tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))
print(classification_report(y_test, pred, target_names=['Benign','Malignant'], digits=4))
print('Confusion matrix [[TN FP],[FN TP]]:', cm.tolist())
results['cls_report'] = classification_report(y_test, pred, target_names=['Benign','Malignant'], digits=4, output_dict=True)
fig, ax = plt.subplots(1,2, figsize=(11,4.3))
ConfusionMatrixDisplay(cm, display_labels=['Benign','Malignant']).plot(ax=ax[0], cmap='Blues', colorbar=False)
ax[0].set_title('Confusion matrix (test set)')
RocCurveDisplay.from_estimator(final, X_test, y_test, ax=ax[1], name='Tuned Random Forest')
ax[1].plot([0,1],[0,1],'k--', lw=1); ax[1].set_title('ROC curve (test set)')
plt.tight_layout(); plt.savefig('fig_eval.png', dpi=150); plt.close()

              precision    recall  f1-score   support

      Benign     0.9600    1.0000    0.9796        72
   Malignant     1.0000    0.9286    0.9630        42

    accuracy                         0.9737       114
   macro avg     0.9800    0.9643    0.9713       114
weighted avg     0.9747    0.9737    0.9735       114

Confusion matrix [[TN FP],[FN TP]]: [[72, 0], [3, 39]]


## 7. Interpretation: which features drive the predictions?

In [15]:
rf = final.named_steps['m']
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
perm = permutation_importance(final, X_test, y_test, n_repeats=20, random_state=RS, scoring='f1')
pimp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
results['top_imp'] = {k: round(float(v),4) for k,v in imp.head(8).items()}
results['top_perm'] = {k: round(float(v),4) for k,v in pimp.head(5).items()}
fig, ax = plt.subplots(figsize=(8,5))
sns.barplot(x=imp.head(12).values, y=imp.head(12).index, color='#4C78A8', ax=ax)
ax.set_title('Top 12 Random Forest feature importances (Gini)'); ax.set_xlabel('Importance')
plt.tight_layout(); plt.savefig('fig_importance.png', dpi=150); plt.close()
imp.head(8)

worst area              0.152915
worst concave points    0.134546
worst radius            0.100371
worst perimeter         0.087629
mean concave points     0.083709
mean perimeter          0.077532
mean radius             0.059437
mean concavity          0.051643
dtype: float64

In [16]:
# Error analysis: which test cases were misclassified and how confident was the model?
err = X_test.copy(); err['true']=y_test; err['pred']=pred; err['p_malignant']=proba.round(3)
errors = err[err.true!=err.pred][['true','pred','p_malignant']]
results['errors'] = errors.reset_index().to_dict('records')
errors

,true,pred,p_malignant
73,1,0,0.150
86,1,0,0.263
385,1,0,0.161


In [17]:
# Threshold analysis: what if we prioritise recall (fewer missed cancers)?
th = []
for t in [0.5,0.4,0.3,0.2]:
    p = (proba>=t).astype(int)
    th.append({'threshold':t,'precision':round(precision_score(y_test,p),4),'recall':round(recall_score(y_test,p),4),
               'FN':int(((p==0)&(y_test==1)).sum()),'FP':int(((p==1)&(y_test==0)).sum())})
th = pd.DataFrame(th); results['threshold']=th.to_dict('records'); th

,threshold,precision,recall,FN,FP
0,0.5,1.0000,0.9286,3,0
1,0.4,0.9750,0.9286,3,1
2,0.3,0.9750,0.9286,3,1
3,0.2,0.8696,0.9524,2,6


In [18]:
# Generalisation gap and stability across different random seeds
train_f1 = f1_score(y_train, final.predict(X_train))
seeds = []
for s in range(10):
    Xa,Xb,ya,yb = train_test_split(X,y,test_size=0.2,stratify=y,random_state=s)
    m = Pipeline([('sc',StandardScaler()),('m',RandomForestClassifier(**{k.replace('m__',''):v for k,v in gs_rf.best_params_.items()}, random_state=RS))]).fit(Xa,ya)
    seeds.append(f1_score(yb, m.predict(Xb)))
results['train_f1']=round(train_f1,4); results['seed_f1_mean']=round(float(np.mean(seeds)),4); results['seed_f1_std']=round(float(np.std(seeds)),4)
results['seed_f1_min']=round(float(np.min(seeds)),4); results['seed_f1_max']=round(float(np.max(seeds)),4)
print('Train F1:', round(train_f1,4), '| 10-split test F1: mean %.4f std %.4f min %.4f max %.4f' % (np.mean(seeds),np.std(seeds),np.min(seeds),np.max(seeds)))

Train F1: 0.9911 | 10-split test F1: mean 0.9255 std 0.0267 min 0.8718 max 0.9639


In [19]:
json.dump(results, open('results.json','w'), indent=2, default=str)
print('Saved results.json and figures.')

Saved results.json and figures.


## 8. Conclusions
* Tuned **SVM** had the best CV F1 (~0.967); tuned **Random Forest** ~0.955. Both reach ~97.4% test accuracy, and the difference between them is within CV noise.
* Hyperparameter tuning gave only a small CV gain: the defaults were already strong on this clean dataset.
* The single test split is optimistic (test F1 0.963); across 10 different random splits the mean F1 was ~0.926 (std ~0.027), so the honest expectation is a little lower.
* The most important predictors are size/shape features of the cell nuclei (worst area, worst concave points, worst radius).
* **Limitations:** small dataset (569 rows), a single hold-out set, and the model is not clinical-grade. It should support, not replace, a pathologist.
* **Future work:** nested cross-validation, calibration, gradient boosting, SHAP explanations, and external validation.